# 01 — Rebuild manifests and enforce the leakage gate

The supplied source split names are treated as candidates. Test cohorts remain fixed. Training/gallery rows are removed for shared patients/groups, studies, image IDs, exact image hashes, exact report hashes, or near-identical perceptual hashes. The combined gallery is cleaned again across datasets.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import pandas as pd
from rerun_code.common import write_json
from rerun_code.leakage_safe_data import (build_manifest, remove_query_conflicts_from_gallery, assert_leakage_free, overlap_audit, cohort_summary, describe_removals, write_jsonl, manifest_sha256)

raw, clean, audits = {}, {}, {}
for dataset in ("mimic", "iuhn"):
    spec = CONFIG["datasets"][dataset]
    train = build_manifest(spec["train"], dataset, "train", compute_phash=True, strict_pairs=True)
    test = build_manifest(spec["test"], dataset, "test", compute_phash=True, strict_pairs=True)
    gallery, removals = remove_query_conflicts_from_gallery(train, test, phash_threshold=CONFIG["phash_threshold"], remove_assumed_patient_groups=True)
    after = assert_leakage_free(gallery, test, phash_threshold=CONFIG["phash_threshold"])
    raw[(dataset, "train")], raw[(dataset, "test")] = train, test
    clean[(dataset, "gallery")], clean[(dataset, "queries")] = gallery, test
    target = PATHS["manifests"] / dataset
    write_jsonl(train, target / "candidate_train.jsonl"); write_jsonl(test, target / "fixed_queries.jsonl")
    write_jsonl(gallery, target / "clean_gallery.jsonl"); write_jsonl(removals, target / "excluded_gallery_records.jsonl")
    audits[dataset] = {"train": cohort_summary(train), "test": cohort_summary(test), "before": overlap_audit(train, test), "removed": describe_removals(removals), "after": after}
    print(dataset, json.dumps(audits[dataset], indent=2))

In [ ]:
combined_candidate = pd.concat([clean[(d, "gallery")] for d in ("mimic", "iuhn")], ignore_index=True)
combined_queries = pd.concat([clean[(d, "queries")] for d in ("mimic", "iuhn")], ignore_index=True)
combined_gallery, combined_removed = remove_query_conflicts_from_gallery(combined_candidate, combined_queries, phash_threshold=CONFIG["phash_threshold"], remove_assumed_patient_groups=True)
combined_after = assert_leakage_free(combined_gallery, combined_queries, phash_threshold=CONFIG["phash_threshold"])
target = PATHS["manifests"] / "combined"
write_jsonl(combined_gallery, target / "clean_gallery.jsonl"); write_jsonl(combined_queries, target / "fixed_queries.jsonl")
write_jsonl(combined_removed, target / "excluded_gallery_records.jsonl")
audits["combined"] = {"removed": describe_removals(combined_removed), "after": combined_after}
checksums = {str(path.relative_to(PATHS["manifests"])): manifest_sha256(path) for path in sorted(PATHS["manifests"].rglob("*.jsonl"))}
write_json(PATHS["manifests"] / "leakage_audit.json", {"audits": audits, "sha256": checksums})
print("LEAKAGE GATE PASSED", json.dumps(checksums, indent=2))